# 04 — Attenuation Experiment

**Goal**: Take well-characterized ZTF sources (high SNR, known type),
progressively attenuate them to sub-threshold levels, and measure
whether TDA can recover the population signal.

This is the key validation: it demonstrates sub-threshold recovery
on *real astrophysical signals*, not synthetic models.

We also compare TDA recovery to classical methods (Lomb-Scargle, stacking)
to quantify where the topological approach adds value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from void.data.ztf import query_objects_by_class, get_batch_light_curves
from void.data.synthetic import generate_noise
from void.embedding.takens import TakensEmbedder
from void.analysis.attenuation import (
    run_attenuation_experiment,
    compare_with_classical,
    attenuate_light_curve,
)
from void.topology.null_model import build_null_distribution
from void.viz.plots import plot_light_curve

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
rng = np.random.default_rng(42)

## 1. Download high-SNR sources

Fetch well-classified RR Lyrae from ZTF as our signal population.

In [ ]:
periodic_objects = query_objects_by_class(
    classifier="lc_classifier",
    class_name="RRL",
    n_objects=50,
    probability_threshold=0.9,
)
oids = periodic_objects.index.tolist()[:30]
print(f"Downloading {len(oids)} RR Lyrae light curves...")
mblcs = get_batch_light_curves(oids, include_forced=True)

# Extract g-band light curves
source_lcs = []
for mblc in mblcs:
    band = "g" if "g" in mblc.bands else (mblc.bands[0] if mblc.bands else None)
    if band:
        lc = mblc[band]
        if lc.peak_snr > 5:  # only keep high-SNR sources
            source_lcs.append(lc)

print(f"Selected {len(source_lcs)} high-SNR sources")
print(f"Mean peak SNR: {np.mean([lc.peak_snr for lc in source_lcs]):.1f}")

## 2. Visualize the attenuation process

Show what happens to a single light curve as we attenuate it.

In [ ]:
if source_lcs:
    example_lc = source_lcs[0]
    factors = [1.0, 0.5, 0.2, 0.1, 0.05]
    fig, axes = plt.subplots(1, len(factors), figsize=(4 * len(factors), 3))

    for ax, f in zip(axes, factors):
        att_lc = attenuate_light_curve(example_lc, f)
        plot_light_curve(att_lc, ax=ax, title=f"factor={f:.2f}")
        ax.text(0.05, 0.95, f"peak SNR={att_lc.peak_snr:.1f}",
                transform=ax.transAxes, fontsize=8, va="top")

    fig.suptitle("Light Curve Attenuation", y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("No source light curves available. Check ALeRCE connection.")

## 3. Build null distribution

In [ ]:
embedder = TakensEmbedder(dimension=3, delay=2)

# Generate noise background
noise_lcs = [
    generate_noise(n_epochs=200, rng=np.random.default_rng(rng.integers(0, 2**63)))
    for _ in range(200)
]

n_total = len(source_lcs) + len(noise_lcs)
print(f"Building null distribution ({n_total} sources per realization)...")
null_dist = build_null_distribution(
    n_realizations=50,
    n_sources_per=n_total,
    n_epochs=200,
    embedder=embedder,
    rng=np.random.default_rng(rng.integers(0, 2**63)),
    verbose=True,
)

## 4. Run the attenuation experiment

Sweep attenuation from 1.0 (original) down to 0.05 (5% of original signal).

In [ ]:
experiment = run_attenuation_experiment(
    source_light_curves=source_lcs,
    noise_light_curves=noise_lcs,
    attenuation_factors=[0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0],
    null_dist=null_dist,
    embedder=embedder,
    source_type="RR Lyrae",
    verbose=True,
)

print(f"\nRecovery threshold: factor = {experiment.recovery_threshold:.2f}")

In [ ]:
# Plot the attenuation curve
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

factors = experiment.factors
h1_vals = [r.total_persistence_h1 for r in experiment.results]
p_vals = [r.p_value for r in experiment.results]
z_vals = [r.z_score for r in experiment.results]
detected = [r.detected for r in experiment.results]

# H1 persistence vs attenuation
ax1.plot(factors, h1_vals, "o-", color="#4C72B0")
ax1.axhline(experiment.null_mean_h1, color="red", linestyle="--", label="Null mean")
ax1.axhspan(experiment.null_mean_h1 - 2 * experiment.null_std_h1,
            experiment.null_mean_h1 + 2 * experiment.null_std_h1,
            color="red", alpha=0.1, label="Null ±2σ")
ax1.set_xlabel("Attenuation Factor")
ax1.set_ylabel("Total H₁ Persistence")
ax1.set_title("Topological Signal vs Attenuation")
ax1.legend(fontsize=8)

# p-value vs attenuation
ax2.semilogy(factors, p_vals, "o-", color="#DD8452")
ax2.axhline(0.05, color="gray", linestyle="--", label="p=0.05")
ax2.set_xlabel("Attenuation Factor")
ax2.set_ylabel("p-value")
ax2.set_title("Statistical Significance")
ax2.legend(fontsize=8)

# Detection outcome
colors = ["green" if d else "red" for d in detected]
ax3.bar(range(len(factors)), [1 if d else 0 for d in detected], color=colors, alpha=0.7)
ax3.set_xticks(range(len(factors)))
ax3.set_xticklabels([f"{f:.2f}" for f in factors], rotation=45)
ax3.set_xlabel("Attenuation Factor")
ax3.set_ylabel("Detected")
ax3.set_title("Detection Outcome")

fig.suptitle(f"Attenuation Experiment — {experiment.source_type} "
             f"({experiment.n_sources} sources)", y=1.04)
fig.tight_layout()
plt.show()

## 5. Comparison with classical methods

At the same attenuation levels, how do Lomb-Scargle periodograms
and simple stacking perform?

In [ ]:
classical = compare_with_classical(
    source_light_curves=source_lcs,
    noise_light_curves=noise_lcs,
    attenuation_factors=[0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0],
    verbose=True,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(classical["factors"], classical["ls_power"], "o-", label="Lomb-Scargle peak power")
ax1.set_xlabel("Attenuation Factor")
ax1.set_ylabel("Mean LS Peak Power")
ax1.set_title("Periodogram Detection")
ax1.legend()

ax2.plot(classical["factors"], classical["stack_snr"], "o-", color="#55A868",
         label="Stacked SNR")
ax2.axhline(3, color="gray", linestyle="--", label="3σ threshold")
ax2.axhline(5, color="gray", linestyle=":", label="5σ threshold")
ax2.set_xlabel("Attenuation Factor")
ax2.set_ylabel("Stacked SNR")
ax2.set_title("Simple Stacking Detection")
ax2.legend()

fig.suptitle("Classical Methods vs Attenuation", y=1.02)
fig.tight_layout()
plt.show()

## Summary

The attenuation experiment answers the key question: **At what signal level
does topological recovery fail, and how does it compare to classical methods?**

Key findings:
1. The recovery threshold (minimum attenuation factor for detection) defines
   the effective sensitivity limit of the TDA pipeline.
2. TDA operates on the *population* level — it can detect a coherent signal
   population even when individual sources are undetectable.
3. Comparison with Lomb-Scargle and stacking reveals where TDA adds value
   (population structure) vs. where classical methods suffice (individual detection).